# Ground-Truth Driver / Non-Driver Gene Retrieval (Unmatched Negatives)

Retrieves the ground-truth driver and non-driver gene sets (protein-coding universe -> driver
exclusion -> mutation-frequency filter -> disease-pathway filter) and saves **four separate
files**:

1. `driver_genes.txt` -- one gene per line, the ground-truth positive (driver) set
2. `non_driver_genes.txt` -- one gene per line, the ground-truth negative (non-driver) set
3. `unlabeled_genes.txt` -- one gene per line, protein-coding genes that ended up with
   neither label (suspected drivers per NCG/IntOGen/Bailey that CGC doesn't confirm, plus
   genes dropped by the mutation-frequency/pathway filters)
4. `gene_labels.csv` -- drivers + non-drivers combined into one labeled table (`label`
   column: 1 = driver, 0 = non-driver; unlabeled genes are not included in this file)

Non-driver genes are **not** length/expression-matched here -- whatever survives the
mutation-frequency and disease-pathway filters is kept as-is as the negative class.


In [ ]:
import os
import random
import sys
from pathlib import Path

import pandas as pd

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

import config
from src.driver_labeling import (
    apply_mutation_frequency_filter,
    apply_pathway_filter,
    build_gene_labels,
    load_driver_genes,
)
from src.gene_universe import build_protein_coding_gene_universe
from src.negative_sampling import build_driver_exclusion_set
from src.pathway_filter import build_disease_pathway_genes

random.seed(config.RANDOM_SEED)
config.DATA_DIR.mkdir(parents=True, exist_ok=True)
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"[CONFIG] USE_MUTATION_FILTER: {config.USE_MUTATION_FILTER}")
print(f"[CONFIG] USE_PATHWAY_FILTER: {config.USE_PATHWAY_FILTER}")


[CONFIG] USE_MUTATION_FILTER: True
[CONFIG] USE_PATHWAY_FILTER: True


## 1. Protein-coding gene universe

Parses the GENCODE GTF and keeps only `gene_type == "protein_coding"`.


In [2]:
all_genes = build_protein_coding_gene_universe(
    "../../data/gencode.v49.basic.annotation.gtf"
)
pd.DataFrame(sorted(all_genes), columns=["gene_name"]).to_csv(
    config.GENCODE_GENES_FILE, index=False
)
print(f"[GENE UNIVERSE] {len(all_genes)} protein-coding genes")


[GENE UNIVERSE] 20070 protein-coding genes


## 2. Ground-truth driver genes and the expanded exclusion set

Positives (ground-truth drivers) come from the COSMIC Cancer Gene Census. The exclusion set
used to derive raw non-driver candidates is broader -- NCG + CGC + IntOGen + Bailey et al. 2018
-- so a gene isn't kept as a "negative" just because it's missing from CGC specifically.


In [3]:
driver_genes = load_driver_genes("../../data/Census_allWed.tsv")
print(f"[DRIVERS] {len(driver_genes)} ground-truth CGC driver genes")

driver_exclusion_set = build_driver_exclusion_set(
    ncg_file=config.NCG_FILE if config.NCG_FILE.exists() else None,
    cgc_file=config.CGC_CENSUS_FILE if config.CGC_CENSUS_FILE.exists() else None,
    intogen_file=config.INTOGEN_FILE if config.INTOGEN_FILE.exists() else None,
    bailey_file=config.BAILEY_FILE if config.BAILEY_FILE.exists() else None,
)

non_drivers = all_genes - driver_exclusion_set
print(
    f"[NON-DRIVERS] {len(non_drivers)} raw candidates "
    "(excluded from NCG/CGC/IntOGen/Bailey)"
)


[DRIVERS] 763 ground-truth CGC driver genes
[NCG] skipped (no file provided)
[CGC] 763 genes
[IntOGen] skipped (no file provided)
[Bailey et al.] skipped (no file provided)
[DRIVER EXCLUSION SET] 763 unique genes across all sources
[NON-DRIVERS] 19326 raw candidates (excluded from NCG/CGC/IntOGen/Bailey)


## 3. Mutation-frequency filter

Excludes any candidate with mutation frequency >= `config.MUTATION_FREQUENCY_THRESHOLD` in any
TCGA cancer type. Skipped entirely if `config.USE_MUTATION_FILTER` is `False`.


In [4]:
if config.USE_MUTATION_FILTER:
    before = len(non_drivers)
    non_drivers = apply_mutation_frequency_filter(
        non_drivers,
        config.MUTATION_FREQUENCY_FILE,
        threshold=config.MUTATION_FREQUENCY_THRESHOLD,
    )
    print(
        f"[MUTATION FILTER] {len(non_drivers)} remaining "
        f"(-{before - len(non_drivers)})"
    )
else:
    print("[MUTATION FILTER] skipped (config.USE_MUTATION_FILTER is False)")


[MUTATION FILTER] 2491 remaining (-16835)


## 4. Disease-pathway filter

Excludes candidates belonging to a Reactome pathway that is a descendant of the top-level
*Disease* pathway. Skipped entirely if `config.USE_PATHWAY_FILTER` is `False`.


In [5]:
if config.USE_PATHWAY_FILTER:
    pathway_genes = build_disease_pathway_genes(
        "../../data/reactome/ReactomePathways.gmt",
        "../../data/reactome/reactome_relations.csv",
    )
    pd.DataFrame(sorted(pathway_genes), columns=["gene"]).to_csv(
        config.DISEASE_PATHWAY_GENES_FILE, index=False
    )
    print(f"[PATHWAY FILTER] {len(pathway_genes)} disease-pathway genes")

    before = len(non_drivers)
    non_drivers = apply_pathway_filter(non_drivers, pathway_genes)
    print(
        f"[PATHWAY FILTER] {len(non_drivers)} remaining "
        f"(-{before - len(non_drivers)})"
    )
else:
    print("[PATHWAY FILTER] skipped (config.USE_PATHWAY_FILTER is False)")


[PATHWAY FILTER] kept 782 disease-related pathways, skipped 2048 unrelated pathways
[PATHWAY FILTER] 2505 disease-pathway genes
[PATHWAY FILTER] 2421 remaining (-70)


## 5. Save four separate files

`driver_genes.txt`, `non_driver_genes.txt`, and `unlabeled_genes.txt` each hold one gene set
on its own (one gene per line); `gene_labels.csv` combines drivers + non-drivers into a
single labeled table for anything downstream that wants the pair together (unlabeled genes
are intentionally left out of that file since they have no `label` value).


In [6]:
# Genes that ended up with neither label: suspected drivers per NCG/IntOGen/Bailey
# that CGC doesn't confirm (so they were excluded from non_drivers, but aren't in
# driver_genes either), plus genes dropped by the mutation-frequency/pathway filters.
unlabeled_genes = all_genes - driver_genes - non_drivers
print(f"[UNLABELED] {len(unlabeled_genes)} genes excluded from both classes")

driver_path = config.PROCESSED_DIR / "driver_genes.txt"
nondriver_path = config.PROCESSED_DIR / "non_driver_genes.txt"
unlabeled_path = config.PROCESSED_DIR / "unlabeled_genes.txt"

driver_path.write_text("\n".join(sorted(driver_genes)) + "\n")
nondriver_path.write_text("\n".join(sorted(non_drivers)) + "\n")
unlabeled_path.write_text("\n".join(sorted(unlabeled_genes)) + "\n")

labels_df = build_gene_labels(driver_genes, non_drivers)
print(labels_df["label"].value_counts())
labels_df.to_csv(config.GENE_LABELS_FILE, index=False)

print(f"[DONE] Drivers ({len(driver_genes)}): {driver_path}")
print(f"[DONE] Non-drivers ({len(non_drivers)}, unmatched): {nondriver_path}")
print(f"[DONE] Unlabeled ({len(unlabeled_genes)}): {unlabeled_path}")
print(f"[DONE] Combined labels (drivers + non-drivers only): {config.GENE_LABELS_FILE}")

labels_df.head()


[UNLABELED] 16905 genes excluded from both classes
label
0    2421
1     763
Name: count, dtype: int64
[DONE] Drivers (763): ../data/processed/driver_genes.txt
[DONE] Non-drivers (2421, unmatched): ../data/processed/non_driver_genes.txt
[DONE] Unlabeled (16905): ../data/processed/unlabeled_genes.txt
[DONE] Combined labels (drivers + non-drivers only): ../data/processed/gene_labels_driver_vs_nondriver.csv


,gene,label
0,ACVR2A,1
1,RPL5,1
2,TCL1A,1
3,MNX1,1
4,NFKBIE,1
